# Chien luoc cua Bai tap 5, doi voi Crypto

## Gan chien luoc nay vao Auto 1 phut

## 1 phut se goi 1 lan, va khi goi se goi Timeframe 1p de tinh ra Buy_Signal/ Sell_Signal

# Viet ham load data

In [1]:
def loaddataBinance_FromTo(symbol, from_date, to_date, timeframe):
	import pandas as pd
	import ccxt
	from datetime import datetime, timedelta

	# Khởi tạo kết nối đến sàn Binance
	exchange = ccxt.binance()

	# Định dạng ngày tháng
	since = exchange.parse8601(from_date + 'T00:00:00Z')
	end = exchange.parse8601(to_date + 'T00:00:00Z')

	# Lấy dữ liệu thị trường từ sàn Binance
	# 1d
	ohlcv = exchange.fetch_ohlcv(symbol, timeframe=timeframe, since=since, limit=1000)

	# Chuyển dữ liệu thành DataFrame
	data = pd.DataFrame(ohlcv, columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
	data['Timestamp'] = pd.to_datetime(data['Timestamp'], unit='ms')         
	data = data.rename(columns={'Timestamp': 'Datetime'})

	return data

# Goi ham tinh toan chien luoc

In [2]:
def schedule_detectSignal():
	import pandas as pd
	import plotly.graph_objects as go
	import redis
	import numpy as np
	from datetime import datetime, timedelta

	# ##############################################Step 0: Các tham số để lấy dữ liệu###############################
	symbol = 'ETHUSDT'
	from_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
	to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
	timeframe = '5m' # 1m

	# ##############################################Step 1: Lấy dữ liệu##############################################
	data = loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)
	# print(data)

	# ##############################################Step 2: Chiến lược##############################################  
	# Đặt thời gian cho MA và ATR
	ma_fast_period = 50 
	ma_slow_period = 200
	atr_period = 14
	atr_threshold = 40 # Xác định ngưỡng biến động thị trường

	import talib
	# Tính toán MA nhanh và MA chậm bằng TA-Lib
	data['MA_Fast'] = talib.SMA(data['Close'], timeperiod=ma_fast_period)
	data['MA_Slow'] = talib.SMA(data['Close'], timeperiod=ma_slow_period)
	# Sử dụng TA-Lib để tính ATR, trong đó True Range (TR) được tính toán tự động
	data['ATR'] = talib.ATR(data['High'], data['Low'], data['Close'], timeperiod=atr_period)

	# Xác định tín hiệu mua
	data['Buy_Signal'] = (data['MA_Fast'] >= data['MA_Slow']) & (data['ATR'] <= atr_threshold)
	# Xác định tín hiệu bán
	data['Sell_Signal'] = (data['MA_Fast'] < data['MA_Slow']) & (data['ATR'] <= atr_threshold)

	print(data)

# Chay tu dong 1 phut chay 1 lan

In [ ]:
from datetime import datetime, timedelta
import schedule
import time
import random

# Danh sách các phút cụ thể bạn muốn hàm được chạy
run_minutes = list(range(0, 60, 5))  # All minutes (0 to 59)
# Thiết lập thời gian chạy cuối cùng để theo dõi
setattr(schedule_detectSignal, 'last_run', None)

while True:
    current_time = datetime.now()
    current_minute = current_time.minute
    current_second = current_time.second

    # Kiểm tra xem giây hiện tại có bằng 0 không
    if current_second == 0:
        # Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
        if current_minute in run_minutes:
            # Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
            last_run = getattr(schedule_detectSignal, 'last_run', None)
            if last_run is None or last_run != current_minute:
                schedule_detectSignal()
                # Lưu lại phút cuối cùng mà hàm đã chạy
                setattr(schedule_detectSignal, 'last_run', current_minute)

               Datetime     Open     High      Low    Close     Volume  \
0   2024-09-10 00:00:00  2359.51  2360.59  2354.20  2359.00   718.3226   
1   2024-09-10 00:05:00  2358.99  2359.20  2353.95  2355.82   498.3274   
2   2024-09-10 00:10:00  2355.82  2356.18  2349.01  2350.01   902.5757   
3   2024-09-10 00:15:00  2350.00  2353.01  2346.75  2351.11  1653.7915   
4   2024-09-10 00:20:00  2351.10  2351.69  2344.00  2345.13  1111.9879   
..                  ...      ...      ...      ...      ...        ...   
995 2024-09-13 10:55:00  2366.86  2371.39  2365.88  2371.39  1275.8446   
996 2024-09-13 11:00:00  2371.38  2371.42  2367.00  2369.26   848.5012   
997 2024-09-13 11:05:00  2369.27  2370.28  2367.07  2368.60  1455.2699   
998 2024-09-13 11:10:00  2368.61  2372.19  2368.60  2371.81   674.8707   
999 2024-09-13 11:15:00  2371.81  2376.22  2367.26  2369.20  1494.5799   

       MA_Fast     MA_Slow       ATR  Buy_Signal  Sell_Signal  
0          NaN         NaN       NaN       Fals